## Feature Engineering: Automating Preprocessing with ColumnTransformer
---

Instead of manually slicing columns, handling missing values, encoding categories, and concatenating arrays back together, we used Scikit-Learn's **`ColumnTransformer`**. This allows us to apply targeted preprocessing pipelines to specific columns simultaneously while maintaining a clean, error-free workflow.

### Preprocessing Architecture

Our transformer executes three distinct transformations in a single parallel step:

* **`tnf1` (Numerical Imputation):** Applies `SimpleImputer()` to the `fever` column to fill in missing values using the mean strategy.
* **`tnf2` (Ordinal Encoding):** Converts the categorical `cough` feature into ordered integers matching our custom ranking: `['Mild', 'Strong']`.
* **`tn3` (One-Hot Encoding):** Transforms nominal variables (`gender` and `city`) into binary dummy columns, using `drop='first'` to prevent multicollinearity and `sparse_output=False` to return a dense array.
* **`remainder='passthrough'`:** Ensures that all unspecified columns (like `age`) are kept intact and passed through to the final output matrix without any modifications.

---

### Implementation & Execution

```python
# 1. Fit and transform the training matrix
X_train_transformed = transformer.fit_transform(X_train)

# 2. Transform the testing matrix (Preventing data leakage)
X_test_transformed = transformer.transform(X_test)
```

### Why this is a Best Practice
1. **No Data Leakage:** It completely eliminates the risk of accidentally running `.fit_transform()` on the test dataset.
2. **Fewer Code Lines:** Replaces over 25 lines of manual array slicing, indexing, and `np.concatenate` calls with 1 clean object configuration.
3. **Pipeline Ready:** This object can now be plugged directly into a Scikit-Learn `Pipeline` alongside a machine learning algorithm for automated end-to-end model deployments.


In [1]:
import pandas as pd
import numpy as np

In [11]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [12]:
df = pd.read_csv('covid_toy.csv')

In [13]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [14]:
# df['cough'].value_counts()
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [15]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [16]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['has_covid']), df['has_covid'],
                                                     test_size=0.2,
                                                     random_state=0)

In [17]:
X_train

,age,gender,fever,cough,city
43,22,Female,99.0,Mild,Bangalore
62,56,Female,104.0,Strong,Bangalore
3,31,Female,98.0,Mild,Kolkata
71,75,Female,104.0,Strong,Delhi
45,72,Male,99.0,Mild,Bangalore
...,...,...,...,...,...
96,51,Female,101.0,Strong,Kolkata
67,65,Male,99.0,Mild,Bangalore
64,42,Male,104.0,Mild,Mumbai
47,18,Female,104.0,Mild,Bangalore


## 1. Without Column Transformer

In [39]:
# Adding simple imputer to fever column
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# Also on test data
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [40]:
# OrdinalEncodin -> cough
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# Also on test data
X_test_cough = oe.transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [41]:
# OneHotEncoding -> gender, city
ohe = OneHotEncoder(drop='first', sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])

# Also on test data
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

X_train_gender_city.shape

(80, 4)

In [42]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever', 'cough', 'city']).values

# Also on test data
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values
X_train_age.shape

(80, 1)

In [44]:
X_train_transformed = np.concatenate((X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1)

# Also on test data
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1)

X_train_transformed.shape

(80, 7)

## 2. With Column Transformer

In [48]:
from sklearn.compose import ColumnTransformer
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tn3', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
], remainder='passthrough')

In [51]:
transformer.fit_transform(X_train).shape

(80, 7)

In [53]:
transformer.transform(X_test).shape

(20, 7)